In [1]:

# COEQWAL — Land Surface Temperature Pipeline
# IMPORTS 
import difflib
import io
import logging
import os
import re
import sys
import textwrap
import uuid
import zipfile
from pathlib import Path

import datacube
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import requests
import rioxarray
import xarray as xr
import yaml
from rasterio.env import Env
from requests.adapters import HTTPAdapter
from rioxarray.merge import merge_arrays
from urllib3.util.retry import Retry


In [2]:

# LOGGING
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("coeqwal_lst")


In [3]:
# CONFIG from YAML 
_DEFAULT_CONFIG = "config.yaml"

def load_config(path: str = _DEFAULT_CONFIG) -> dict:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Config file not found: {p.resolve()}")
    with open(p) as f:
        return yaml.safe_load(f)

# Allow override via CLI:  python script.py --config other.yaml
_cfg_path = _DEFAULT_CONFIG
if "--config" in sys.argv:
    idx = sys.argv.index("--config")
    if idx + 1 < len(sys.argv):
        _cfg_path = sys.argv[idx + 1]

CFG = load_config(_cfg_path)

# --- Unpack into module-level constants (still ONE place to change: the YAML) ---
INPUTS_DIR  = Path("inputs")
OUTPUTS_DIR = Path("outputs")

AOI_FILE    = Path(CFG["aoi_file"])
TARGETS_CSV = Path(CFG["targets_csv"])

OUT_COVERAGE_PAIRS_CSV  = Path(CFG["outputs"]["coverage_csv"])
OUT_EFFECTIVE_TILES_CSV = Path(CFG["outputs"]["effective_tiles_csv"])

OUTPUT_CRS = CFG["output_crs"]
RESOLUTION = tuple(CFG["resolution"])

WRS_PATH = CFG["wrs_path"]
WRS_ROWS = set(CFG["wrs_rows"])

SEARCH_WINDOW_DAYS = CFG["search_window_days"]
CLOUD_COVER_MAX    = CFG.get("cloud_cover_max")    # None → disabled

USE_CLEAR       = CFG["use_clear"]
WATER_ONLY      = CFG["water_only"]
USE_RADSAT_MASK = CFG["use_radsat_mask"]

SENSOR_PREF = CFG["sensor_preference"]
NODATA_OUT  = CFG["nodata_out"]


In [4]:
# AWS + GDAL environment
_aws = CFG.get("aws", {})
os.environ["AWS_REQUEST_PAYER"]  = _aws.get("request_payer", "requester")
os.environ["AWS_DEFAULT_REGION"] = _aws.get("default_region", "us-west-2")
os.environ["AWS_REGION"]         = _aws.get("default_region", "us-west-2")
os.environ.pop("AWS_NO_SIGN_REQUEST", None)

os.environ["GDAL_DISABLE_READDIR_ON_OPEN"]    = "YES"
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".tif,.TIF,.xml,.XML"
os.environ["VSI_CACHE"]      = "TRUE"
os.environ["VSI_CACHE_SIZE"] = str(128 * 1024 * 1024)

logger.info("AWS_ACCESS_KEY_ID present?: %s", "AWS_ACCESS_KEY_ID" in os.environ)
logger.info("AWS_REQUEST_PAYER: %s", os.environ.get("AWS_REQUEST_PAYER"))
logger.info("AWS_DEFAULT_REGION: %s", os.environ.get("AWS_DEFAULT_REGION"))


2026-05-05 23:26:14  INFO      AWS_ACCESS_KEY_ID present?: True
2026-05-05 23:26:14  INFO      AWS_REQUEST_PAYER: requester
2026-05-05 23:26:14  INFO      AWS_DEFAULT_REGION: us-west-2


In [5]:
#  CHUNK 0 — Utilities

# --- Requests session with retries (change 7) ---
def _build_session() -> requests.Session:
    s = requests.Session()
    retry = Retry(total=3, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.mount("http://",  HTTPAdapter(max_retries=retry))
    s.headers.update({"User-Agent": "Mozilla/5.0"})
    return s

SESSION = _build_session()

# --- AOI helpers ---
_AOI_CACHE_4326 = None

def load_aoi_4326(aoi_path: Path = AOI_FILE) -> gpd.GeoDataFrame:
    global _AOI_CACHE_4326
    if _AOI_CACHE_4326 is not None:
        return _AOI_CACHE_4326

    if not aoi_path.exists():
        raise FileNotFoundError(f"AOI file not found: {aoi_path.resolve()}")

    aoi = gpd.read_file(aoi_path)
    if aoi.crs is None:
        raise ValueError("AOI has no CRS defined (aoi.crs is None).")

    if aoi.crs.to_string() != "EPSG:4326":
        aoi = aoi.to_crs("EPSG:4326")

    if (~aoi.is_valid).any():
        aoi["geometry"] = aoi.geometry.buffer(0)
        aoi = aoi[aoi.geometry.notnull()].copy()

    if len(aoi) > 1:
        aoi = aoi.dissolve().reset_index(drop=True)

    _AOI_CACHE_4326 = aoi
    return aoi

def get_bbox_wgs84(aoi_path: Path = AOI_FILE) -> dict:
    """bbox in WGS84 for datacube.find_datasets/load."""
    aoi = load_aoi_4326(aoi_path)
    minx, miny, maxx, maxy = aoi.total_bounds
    bbox = {"x": (minx, maxx), "y": (miny, maxy)}
    logger.info("AOI bbox (EPSG:4326): %s", bbox)
    return bbox

def get_aoi_in_crs(dst_crs: str = OUTPUT_CRS, aoi_path: Path = AOI_FILE) -> gpd.GeoDataFrame:
    """AOI geometry reprojected to dst_crs."""
    return load_aoi_4326(aoi_path).to_crs(dst_crs)

# --- Time helpers ---
def _date_range(center_date, days):
    c = pd.Timestamp(center_date).normalize()
    return (c - pd.Timedelta(days=days), c + pd.Timedelta(days=days + 1))

def _as_utc_naive(ts):
    ts = pd.Timestamp(ts)
    if ts.tz is not None:
        return ts.tz_convert("UTC").tz_localize(None)
    return ts

def _extract_scene_datetime(ds):
    md = getattr(ds, "metadata_doc", None) or {}
    if isinstance(md, dict):
        dt = (md.get("properties", {}) or {}).get("datetime", None)
        if dt:
            try:
                return _as_utc_naive(pd.to_datetime(dt))
            except Exception as e:                                   # change 6
                logger.debug("Could not parse datetime %r: %s", dt, e)
    ct = getattr(ds, "center_time", None)
    if ct is not None:
        try:
            return _as_utc_naive(pd.to_datetime(ct))
        except Exception as e:                                       # change 6
            logger.debug("Could not parse center_time %r: %s", ct, e)
    try:
        return _as_utc_naive(pd.to_datetime(ds.metadata.time))
    except Exception:
        return None

def _extract_cloud_cover(ds):
    md = getattr(ds, "metadata_doc", None) or {}
    if isinstance(md, dict):
        cc = (md.get("properties", {}) or {}).get("eo:cloud_cover", None)
        if cc is None:
            return None
        try:
            return float(cc)
        except Exception:
            return None
    return None

# --- WRS path/row extraction ---
_SCENEID_PR = re.compile(r".*_(\d{3})(\d{3})_.*")

def _extract_wrs_path_row(ds):
    """Returns (path:int|None, row:int|None)."""
    md = getattr(ds, "metadata_doc", None) or {}
    props = {}
    if isinstance(md, dict):
        props = (md.get("properties", {}) or {})

    for kpath, krow in [
        ("landsat:wrs_path", "landsat:wrs_row"),
        ("wrs_path", "wrs_row"),
        ("landsat:wrs_path", "wrs_row"),
        ("wrs_path", "landsat:wrs_row"),
    ]:
        p = props.get(kpath)
        r = props.get(krow)
        try:
            p = int(p) if p is not None else None
            r = int(r) if r is not None else None
        except Exception:
            p, r = None, None
        if p is not None and r is not None:
            return p, r

    scene_id = props.get("landsat:scene_id") or props.get("scene_id")
    if scene_id:
        m = _SCENEID_PR.match(str(scene_id))
        if m:
            try:
                return int(m.group(1)), int(m.group(2))
            except Exception as e:                                   # change 6
                logger.debug("Could not parse scene_id %r: %s", scene_id, e)

    return None, None

# --- Landsat ST product discovery ---
def _product_priority(product_name: str, sensor_pref: list) -> int:  # change 4
    """Tiebreaker: lower = preferred."""
    p = product_name.lower()
    for i, key in enumerate(sensor_pref, start=1):
        if key in p:
            return i
    return 999

def list_landsat_st_products(dc: datacube.Datacube, sensor_pref: list) -> list:  # change 4
    prods = dc.list_products()
    names = list(prods.index.astype(str))

    cand = []
    for name in names:
        n = name.lower()
        if ("landsat" in n) and ("c2" in n) and ("l2" in n) and ("st" in n):
            cand.append(name)

    if not cand:
        meas = dc.list_measurements()
        for name in names:
            try:
                mnames = set(meas.loc[name].index.astype(str).str.lower())
                if "qa_pixel" in mnames and ("lwir11" in mnames or "st" in mnames):
                    cand.append(name)
            except Exception as e:                                   # change 6
                logger.debug("Skipping product %s during measurement scan: %s", name, e)

    return sorted(set(cand), key=lambda x: (_product_priority(x, sensor_pref), x))

# --- QA mask + ST conversion ---
BIT_DILATED_CLOUD = 1
BIT_CIRRUS        = 2
BIT_CLOUD         = 3
BIT_CLOUD_SHADOW  = 4
BIT_SNOW          = 5
BIT_CLEAR         = 6
BIT_WATER         = 7

def _bit_is_set(x, bit):
    return (x.astype("uint16") & (1 << bit)) > 0

def build_good_mask(qa_pixel: xr.DataArray,                     # change 4
                    water_only: bool, use_clear: bool) -> xr.DataArray:
    bad = (
        _bit_is_set(qa_pixel, BIT_DILATED_CLOUD) |
        _bit_is_set(qa_pixel, BIT_CIRRUS) |
        _bit_is_set(qa_pixel, BIT_CLOUD) |
        _bit_is_set(qa_pixel, BIT_CLOUD_SHADOW) |
        _bit_is_set(qa_pixel, BIT_SNOW)
    )
    good = ~bad
    if use_clear:
        good = good & _bit_is_set(qa_pixel, BIT_CLEAR)
    if water_only:
        good = good & _bit_is_set(qa_pixel, BIT_WATER)
    return good


# --- Measurement resolver ---
def _aliases_lower(meas_def: dict) -> set:
    als = meas_def.get("aliases", []) or []
    return {str(a).lower() for a in als}

def resolve_measurements_from_definition(dc: datacube.Datacube, product: str):
    """Returns: (st_name, qa_pixel_name, qa_radsat_name_or_None)"""
    prod = dc.index.products.get_by_name(product)
    if prod is None:
        raise ValueError(f"Product not found in ODC index: {product}")

    meas_defs = (prod.definition or {}).get("measurements", []) or []
    if not meas_defs:
        raise ValueError(f"No measurements in product definition for: {product}")

    # 1) Find ST
    st_candidates = []
    for m in meas_defs:
        name = str(m.get("name", ""))
        if not name:
            continue
        als = _aliases_lower(m)
        is_st = (
            ("st" in als) or
            ("surface_temperature" in als) or
            any(a.startswith("st_b") for a in als)
        )
        if is_st:
            rank = 0
            if "st" in als: rank += 100
            if "surface_temperature" in als: rank += 90
            if any(a.startswith("st_b") for a in als): rank += 80
            if "lwir" in name.lower(): rank += 10
            st_candidates.append((rank, name, als))

    if not st_candidates:
        for m in meas_defs:
            name = str(m.get("name", ""))
            units = str(m.get("units", "")).lower()
            if "kelvin" in units and "qa" not in name.lower():
                st_candidates.append((1, name, _aliases_lower(m)))

    if not st_candidates:
        raise KeyError(f"Could not resolve ST measurement for product={product}")

    st_candidates.sort(reverse=True)
    st_name = st_candidates[0][1]

    # 2) qa_pixel
    qa_pixel_name = None
    for m in meas_defs:
        name = str(m.get("name", ""))
        if not name:
            continue
        als = _aliases_lower(m)
        if (name.lower() == "qa_pixel") or ("qa_pixel" in als) or ("pixel_quality" in als) or ("pq" in als):
            qa_pixel_name = name
            break
    if qa_pixel_name is None:
        raise KeyError(f"Could not resolve qa_pixel for product={product}")

    # 3) qa_radsat (optional)
    qa_radsat_name = None
    for m in meas_defs:
        name = str(m.get("name", ""))
        if not name:
            continue
        als = _aliases_lower(m)
        if (name.lower() == "qa_radsat") or ("qa_radsat" in als) or ("radiometric_saturation" in als) or ("radsat" in als):
            qa_radsat_name = name
            break

    return st_name, qa_pixel_name, qa_radsat_name


In [6]:
#  CHUNK 1 — Outflow column detector

def find_outflow_col(df):
    """Returns the ORIGINAL name of the column that contains Net Delta Outflow."""
    def _norm(s: str) -> str:
        s = str(s).strip().upper()
        s = re.sub(r"\s+", "_", s)
        s = re.sub(r"[^A-Z0-9_]", "", s)
        return s

    ALIASES = {
        "NDOI", "NDOI_CFS", "QOUT",
        "OUT1", "OUT", "OUT2",
        "OUTFLOW", "OUTFLOW_CFS",
        "NET_DELTA_OUTFLOW", "NET_DELTA_OUTFLOW_INDEX",
        "NETDELTAOUTFLOW", "NETDELTAOUTFLOWINDEX",
    }

    norm_map = {c: _norm(c) for c in df.columns}

    exact = [orig for orig, n in norm_map.items() if n in ALIASES]
    if exact:
        return exact[0]

    pat = re.compile(
        r"^(NDOI(_CFS)?|QOUT|OUT1|OUT|OUTFLOW(_CFS)?|NET_?DELTA_?OUTFLOW(_INDEX)?)$"
    )
    for orig, n in norm_map.items():
        if pat.match(n):
            return orig

    choices = list(ALIASES)
    for orig, n in norm_map.items():
        if difflib.get_close_matches(n, choices, n=1, cutoff=0.8):
            return orig

    raise KeyError(f"Outflow column not found. Available: {list(df.columns)}")



In [7]:
#  CHUNK 2 — Dayflow downloader  (changes 5, 6, 7)

API_PACKAGE_SHOW = "https://data.cnra.ca.gov/api/3/action/package_show"
API_DS           = "https://data.cnra.ca.gov/api/3/action/datastore_search"
API_RSRC_SHOW    = "https://data.cnra.ca.gov/api/3/action/resource_show"
PACKAGE_ID       = "dayflow"

DAYFLOW_OUT_CSV  = Path(CFG["outputs"]["dayflow_csv"])

def build_date_series(df: pd.DataFrame) -> pd.Series:
    cols = {str(c).strip().lower(): c for c in df.columns}

    if "date" in cols:
        s = pd.to_datetime(df[cols["date"]], errors="coerce")
        if s.notna().any():
            return s

    has = {k: v for k, v in cols.items() if k in ("year", "month", "day")}
    if {"year", "month", "day"}.issubset(has):
        return pd.to_datetime(
            dict(year=df[has["year"]], month=df[has["month"]], day=df[has["day"]]),
            errors="coerce",
        )

    raise KeyError(f"Cannot find Date or (Year,Month,Day). Headers: {list(df.columns)}")

# --- change 7: paginated datastore fetch ---
def get_all_datastore_records(rid: str, session: requests.Session) -> pd.DataFrame | None:
    records, offset = [], 0
    batch_size = 50000
    while True:
        js = session.get(
            API_DS,
            params={"resource_id": rid, "limit": batch_size, "offset": offset},
            timeout=60,
        ).json()
        if not js.get("success") or "records" not in js.get("result", {}):
            break
        batch = js["result"]["records"]
        records.extend(batch)
        if len(batch) < batch_size:
            break
        offset += batch_size
    return pd.DataFrame(records) if records else None

def _is_results_resource(r: dict) -> bool:
    name = str(r.get("name") or r.get("title") or "").lower()
    fmt  = str(r.get("format") or "").lower()
    if "dayflow results" not in name:
        return False
    if any(bad in name for bad in ["monthly", "totals", "comments", "documentation", "doc", "pdf"]):
        return False
    return (fmt in ["csv", "xlsx", "excel"] or fmt == "")

def _infer_year(r: dict):
    txt = f"{r.get('name','')} {r.get('title','')}"
    m = re.search(r"(19\d{2}|20\d{2})", txt)
    return int(m.group(1)) if m else -1

def download_dayflow(session: requests.Session):
    logger.info("Discovering Dayflow resources from CNRA CKAN…")

    pkg = session.get(API_PACKAGE_SHOW, params={"id": PACKAGE_ID}, timeout=60).json()
    if not pkg.get("success"):
        raise RuntimeError(pkg.get("error") or "package_show failed")

    resources = pkg["result"].get("resources", [])
    if not resources:
        raise RuntimeError("No resources found in CKAN package 'dayflow'")

    results_res = [r for r in resources if _is_results_resource(r)]
    if not results_res:
        results_res = [r for r in resources if "results" in str(r.get("name","")).lower()]

    logger.info("Found %d 'Dayflow Results' resources", len(results_res))
    results_res = sorted(results_res, key=lambda r: (_infer_year(r), str(r.get("name",""))))

    frames = []
    for r in results_res:
        rid  = r.get("id")
        name = r.get("name") or r.get("title") or rid
        fmt  = str(r.get("format") or "").upper()

        logger.info("Processing: %s  [format=%s]", name, fmt or "UNKNOWN")

        # A) Try DataStore with pagination (change 7)
        df_raw = get_all_datastore_records(rid, session)

        # B) Direct download fallback
        if df_raw is None:
            meta = session.get(API_RSRC_SHOW, params={"id": rid}, timeout=60).json()
            if not meta.get("success"):
                logger.warning("resource_show failed for %s", name)    # change 6
                continue

            url = meta["result"].get("url")
            if not url:
                logger.warning("No URL for %s", name)
                continue

            raw = session.get(url, timeout=180).content

            try:
                df_raw = pd.read_excel(io.BytesIO(raw), engine="openpyxl")
            except Exception:
                try:
                    df_raw = pd.read_csv(io.BytesIO(raw))
                except Exception as e:                               # change 6
                    logger.warning("Could not read %s as Excel/CSV: %s", name, e)
                    continue

        try:
            date_series = build_date_series(df_raw)
            outcol = find_outflow_col(df_raw)
        except Exception as e:                                       # change 6
            logger.warning("Unexpected headers in %s; skipping: %s", name, e)
            logger.debug("Headers: %s", list(df_raw.columns))
            continue

        df = (
            pd.DataFrame({
                "Date": pd.to_datetime(date_series, errors="coerce"),
                "NDOI": pd.to_numeric(df_raw[outcol], errors="coerce"),
            })
            .dropna(subset=["Date"])
            .sort_values("Date")
        )

        frames.append(df)
        logger.info("  Rows kept: %s (max Date: %s)", f"{len(df):,}", df["Date"].max().date())

    if not frames:
        sys.exit("No Dayflow Results resources could be processed.")

    all_df = (
        pd.concat(frames, ignore_index=True)
        .drop_duplicates("Date")
        .sort_values("Date")
    )

    os.makedirs(DAYFLOW_OUT_CSV.parent, exist_ok=True)
    all_df.to_csv(DAYFLOW_OUT_CSV, index=False)
    logger.info("Dayflow CSV: %s — rows: %s — last date: %s",
                DAYFLOW_OUT_CSV, f"{len(all_df):,}", all_df["Date"].max())

download_dayflow(SESSION)


2026-05-05 23:26:15  INFO      Discovering Dayflow resources from CNRA CKAN…
2026-05-05 23:26:15  INFO      Found 14 'Dayflow Results' resources
2026-05-05 23:26:15  INFO      Processing: Dayflow Results 1929 - 1939  [format=CSV]
2026-05-05 23:26:16  INFO        Rows kept: 3,744 (max Date: 1939-12-31)
2026-05-05 23:26:16  INFO      Processing: Dayflow Results 1940 - 1949  [format=CSV]
2026-05-05 23:26:17  INFO        Rows kept: 3,653 (max Date: 1949-12-31)
2026-05-05 23:26:17  INFO      Processing: Dayflow Results 1950 - 1955  [format=CSV]
2026-05-05 23:26:17  INFO        Rows kept: 2,099 (max Date: 1955-09-30)
2026-05-05 23:26:17  INFO      Processing: Dayflow Results 1956 - 1969  [format=CSV]
2026-05-05 23:26:18  INFO        Rows kept: 5,114 (max Date: 1969-09-30)
2026-05-05 23:26:18  INFO      Processing: Dayflow Results 1970 - 1983  [format=CSV]
2026-05-05 23:26:19  INFO        Rows kept: 5,113 (max Date: 1983-09-30)
2026-05-05 23:26:19  INFO      Processing: Dayflow Results 1984 -

In [8]:

#  CHUNK 3 — Water Year Type

WYT_RID = "105614f4-c71d-4191-b1f9-ea510afd8b62"
WYT_API = "https://data.ca.gov/api/3/action/datastore_search"
WYT_OUT_CSV = Path(CFG["outputs"]["wyt_csv"])

def download_wyt(session: requests.Session):
    records, start, rows = [], 0, 50000
    while True:
        js = session.get(
            WYT_API,
            params={"resource_id": WYT_RID, "limit": rows, "offset": start},
            timeout=60,
        ).json()
        if not js.get("success"):
            raise RuntimeError(js.get("error"))
        recs = js["result"]["records"]
        records.extend(recs)
        if len(recs) < rows:
            break
        start += rows

    long = pd.DataFrame(records)
    long.columns = [c.strip() for c in long.columns]

    wide = (long.pivot(index="WY", columns="Area", values="WYT")
                .reset_index()
                .rename(columns={
                    "Sacramento Valley":  "Sac_Type",
                    "San Joaquin Valley": "SJV_Type"}))
    wide["WY"] = wide["WY"].astype(int)

    os.makedirs(WYT_OUT_CSV.parent, exist_ok=True)
    wide.to_csv(WYT_OUT_CSV, index=False)
    logger.info("Water Year Type CSV: %s — rows: %d", WYT_OUT_CSV, len(wide))

download_wyt(SESSION)


2026-05-05 23:26:24  INFO      Water Year Type CSV: outputs/water_year_type.csv — rows: 124


In [9]:
#  CHUNK 4 — Join Dayflow + WYT

DAYFLOW_WYT_CSV = Path(CFG["outputs"]["dayflow_wyt_csv"])

def join_dayflow_wyt():
    if not DAYFLOW_OUT_CSV.exists():
        raise SystemExit(f"Missing {DAYFLOW_OUT_CSV} — run chunk 2 first.")
    if not WYT_OUT_CSV.exists():
        raise SystemExit(f"Missing {WYT_OUT_CSV} — run chunk 3 first.")

    df_day = pd.read_csv(DAYFLOW_OUT_CSV, parse_dates=["Date"])
    df_wyt = pd.read_csv(WYT_OUT_CSV)

    df_day = df_day.dropna(subset=["Date"]).copy()
    df_day["Date"] = pd.to_datetime(df_day["Date"]).dt.normalize()
    df_day["NDOI"] = pd.to_numeric(df_day["NDOI"], errors="coerce")
    df_day["WY"]   = (df_day["Date"] + pd.DateOffset(months=3)).dt.year.astype(int)

    def normalize_wyt_col(s: pd.Series) -> pd.Series:
        m = {"WET": "W", "ABOVE NORMAL": "AN", "BELOW NORMAL": "BN", "DRY": "D", "CRITICAL": "C"}
        out = s.astype(str).str.strip().str.upper()
        return out.map(m).fillna(out)

    df_wyt["WY"] = pd.to_numeric(df_wyt["WY"], errors="coerce").astype("Int64")
    df_wyt = df_wyt.dropna(subset=["WY"]).copy()
    df_wyt["WY"]       = df_wyt["WY"].astype(int)
    df_wyt["Sac_Type"] = normalize_wyt_col(df_wyt["Sac_Type"])
    df_wyt["SJV_Type"] = normalize_wyt_col(df_wyt["SJV_Type"])

    joined = df_day.merge(df_wyt, on="WY", how="left")
    joined = joined[["WY", "NDOI", "Sac_Type", "SJV_Type", "Date"]].copy()
    joined["Date"] = joined["Date"].dt.strftime("%Y-%m-%d")

    os.makedirs(DAYFLOW_WYT_CSV.parent, exist_ok=True)
    joined.to_csv(DAYFLOW_WYT_CSV, index=False)
    logger.info("Dayflow+WYT CSV: %s — rows: %s", DAYFLOW_WYT_CSV, f"{len(joined):,}")

join_dayflow_wyt()


2026-05-05 23:26:24  INFO      Dayflow+WYT CSV: outputs/dayflow_wyt_daily.csv — rows: 35,064


In [10]:
#  CHUNK 5 — Interactive NDOI + WYT query

def interactive_date_query():
    if not DAYFLOW_WYT_CSV.exists():
        sys.exit(f"{DAYFLOW_WYT_CSV} not found — run chunk 4 first!")

    df = pd.read_csv(DAYFLOW_WYT_CSV, parse_dates=["Date"])
    df = df[["WY", "NDOI", "Sac_Type", "SJV_Type", "Date"]].copy()

    min_wy, max_wy = int(df["WY"].min()), int(df["WY"].max())
    min_yr = df["Date"].min().year
    max_yr = df["Date"].max().year

    print(textwrap.dedent(f"""
        WATER-YEAR TYPE (WYT)
          W=Wet  AN=Above Normal  BN=Below Normal  D=Dry  C=Critical
          Data: {min_wy}–{max_wy}

        NDOI (Net Delta Outflow Index)
          Daily net freshwater outflow (cfs).
          Data: Oct/01/{min_yr} – Sep/30/{max_yr}
    """))

    while True:
        try:
            sac = input("Sacramento WYT [W/AN/BN/D/C] (blank=any): ").strip().upper() or None
            sjv = input("San Joaquin WYT [W/AN/BN/D/C] (blank=any): ").strip().upper() or None

            min_txt = input("Min NDOI (cfs) [blank=no limit]: ").strip()
            max_txt = input("Max NDOI (cfs) [blank=no limit]: ").strip()
            min_ndoi = float(min_txt) if min_txt else -1e12
            max_ndoi = float(max_txt) if max_txt else  1e12
            if min_ndoi > max_ndoi:
                min_ndoi, max_ndoi = max_ndoi, min_ndoi

            df_f = df[
                (df["NDOI"].between(min_ndoi, max_ndoi, inclusive="both")) &
                (True if sac is None else df["Sac_Type"].astype(str).str.strip().str.upper() == sac) &
                (True if sjv is None else df["SJV_Type"].astype(str).str.strip().str.upper() == sjv)
            ].copy().sort_values("Date")

            print(f"\nMatches: {len(df_f):,} days")

            if not df_f.empty:
                os.makedirs("outputs", exist_ok=True)
                df_f[["WY","NDOI","Sac_Type","SJV_Type","Date"]].to_csv(
                    "outputs/match_results.csv", index=False)
                print("Saved ALL matches to outputs/match_results.csv")
                print("\nAll matching rows:\n")
                print(df_f[["WY","NDOI","Sac_Type","SJV_Type","Date"]].to_string(index=False))

                choice = input(
                    "\nPick dates (all / none / comma-list / start:end): "
                ).strip().lower()

                chosen = df_f.copy()
                if choice == "none":
                    chosen = chosen.iloc[0:0]
                elif choice in ("all", ""):
                    pass
                elif ":" in choice:
                    try:
                        a_str, b_str = choice.split(":", 1)
                        a = pd.to_datetime(a_str).date()
                        b = pd.to_datetime(b_str).date()
                        if a > b: a, b = b, a
                        mask = (chosen["Date"].dt.date >= a) & (chosen["Date"].dt.date <= b)
                        chosen = chosen.loc[mask]
                    except Exception as e:
                        print(f"Could not parse range; keeping all. {e}")
                else:
                    want, misses = [], []
                    all_days = set(df_f["Date"].dt.date)
                    for tok in choice.split(","):
                        tok = tok.strip()
                        if not tok:
                            continue
                        try:
                            d = pd.to_datetime(tok).date()
                            (want if d in all_days else misses).append(tok)
                        except Exception:
                            misses.append(tok)
                    chosen = (chosen[chosen["Date"].dt.date.isin(pd.to_datetime(want).date)]
                              if want else chosen.iloc[0:0])
                    if misses:
                        print("Ignored (not in matches):", ", ".join(misses))

                os.makedirs("inputs", exist_ok=True)
                chosen.sort_values("Date").to_csv("inputs/target_dates.csv", index=False)
                print(f"Saved selected dates to inputs/target_dates.csv — {len(chosen)} rows")
                if len(chosen):
                    print("\nSelected dates (first 20):")
                    print(chosen.head(20)[["WY","NDOI","Sac_Type","SJV_Type","Date"]].to_string(index=False))
                else:
                    print("\nNo dates selected.")
            else:
                print("No dates satisfy those conditions.")

        except Exception as e:
            print(f"Error: {e}")

        again = input("\nRefine the search? (y/n): ").strip().lower()
        if again != "y":
            break

interactive_date_query()


WATER-YEAR TYPE (WYT)
  W=Wet  AN=Above Normal  BN=Below Normal  D=Dry  C=Critical
  Data: 1930–2025

NDOI (Net Delta Outflow Index)
  Daily net freshwater outflow (cfs).
  Data: Oct/01/1929 – Sep/30/2025



Sacramento WYT [W/AN/BN/D/C] (blank=any):  10000
San Joaquin WYT [W/AN/BN/D/C] (blank=any):  10050
Min NDOI (cfs) [blank=no limit]:  W
Max NDOI (cfs) [blank=no limit]:  W


Error: could not convert string to float: 'W'



Refine the search? (y/n):  y
Sacramento WYT [W/AN/BN/D/C] (blank=any):  w
San Joaquin WYT [W/AN/BN/D/C] (blank=any):  w
Min NDOI (cfs) [blank=no limit]:  10000
Max NDOI (cfs) [blank=no limit]:  10050



Matches: 12 days
Saved ALL matches to outputs/match_results.csv

All matching rows:

  WY    NDOI Sac_Type SJV_Type       Date
1938 10039.0        W        W 1938-07-26
1952 10035.0        W        W 1951-11-07
1975 10043.0        W        W 1975-07-01
1975 10003.0        W        W 1975-08-31
1995 10040.0        W        W 1994-12-12
1996 10029.0        W        W 1996-06-25
1997 10046.0        W        W 1997-07-16
2006 10008.0        W        W 2006-07-26
2011 10039.0        W        W 2011-07-22
2017 10019.0        W        W 2017-08-11
2019 10029.0        W        W 2018-10-06
2023 10018.0        W        W 2023-09-18



Pick dates (all / none / comma-list / start:end):  2017-08-11:2023-09-18


Saved selected dates to inputs/target_dates.csv — 3 rows

Selected dates (first 20):
  WY    NDOI Sac_Type SJV_Type       Date
2017 10019.0        W        W 2017-08-11
2019 10029.0        W        W 2018-10-06
2023 10018.0        W        W 2023-09-18



Refine the search? (y/n):  n


In [27]:
#  CHUNK 6 — Pair coverage check
CLOUD_COVER_AOI_MAX = 10

def find_candidates_for_date(dc, st_products, bbox, t0, t1,
                             wrs_path, wrs_rows, cloud_cover_max):
    """Stage 1: fast filter using scene-level metadata cloud cover."""
    out = []
    for prod in st_products:
        dss = dc.find_datasets(product=prod, time=(t0, t1), **bbox)
        for ds in dss:
            dt = _extract_scene_datetime(ds)
            if dt is None:
                continue
            scene_day = _as_utc_naive(dt).normalize()
            path, row = _extract_wrs_path_row(ds)
            if path is None or row is None:
                continue
            if int(path) != int(wrs_path):
                continue
            if int(row) not in wrs_rows:
                continue
            cc = _extract_cloud_cover(ds)
            # Stage 1: coarse filter (full-scene metadata)
            if (cloud_cover_max is not None) and (cc is not None) and (cc > float(cloud_cover_max)):
                continue
            out.append({
                "Product": prod,
                "SceneDate": scene_day.date().isoformat(),
                "SceneDT": scene_day,
                "WRS_PATH": int(path),
                "WRS_ROW": int(row),
                "CloudCover": cc,
                "ODC_id": str(ds.id),
            })
    return out
 
 
def _compute_aoi_cloud_fraction(dc, ds_id, product, aoi_out,
                                 output_crs, resolution):
    """Download QA_PIXEL for one dataset, clip to AOI, return cloud+shadow fraction."""
    try:
        ds_obj = dc.index.datasets.get(uuid.UUID(str(ds_id)))
        if ds_obj is None:
            return None
 
        # Resolve QA band name for this product
        _, qa_pixel_band, _ = resolve_measurements_from_definition(dc, product)
 
        with Env(AWS_REQUEST_PAYER="requester", GDAL_DISABLE_READDIR_ON_OPEN="YES"):
            loaded = dc.load(
                datasets=[ds_obj],
                measurements=[qa_pixel_band],
                output_crs=output_crs,
                resolution=resolution,
                group_by="solar_day",
                skip_broken_datasets=True,
            )
 
        if ("time" not in loaded.dims) or (loaded.time.size == 0):
            return None
 
        qa = loaded[qa_pixel_band].isel(time=0)
        qa = qa.rio.write_crs(output_crs)
        qa_clip = qa.rio.clip(aoi_out.geometry, aoi_out.crs, drop=True)
 
        # Cloud OR cloud shadow OR cirrus = "bad"
        cloud  = _bit_is_set(qa_clip, BIT_CLOUD)
        shadow = _bit_is_set(qa_clip, BIT_CLOUD_SHADOW)
        cirrus = _bit_is_set(qa_clip, BIT_CIRRUS)
        bad = cloud | shadow | cirrus
 
        total = float(qa_clip.size)
        if total == 0:
            return None
        return float(bad.sum().values) / total
 
    except Exception as e:
        logger.debug("Could not compute AOI cloud fraction for %s: %s", ds_id, e)
        return None
 
 
def filter_pairs_by_aoi_cloud(pairs, dc, aoi_out, output_crs,
                               resolution, cloud_cover_aoi_max):
    """Stage 2: precise filter — download QA_PIXEL, clip to AOI, check cloud %."""
    if cloud_cover_aoi_max is None:
        return pairs  # disabled
 
    threshold = float(cloud_cover_aoi_max) / 100.0
    kept = []
 
    for p in pairs:
        product = p["Product"]
 
        frac33 = _compute_aoi_cloud_fraction(
            dc, p["ODC_id_row33"], product, aoi_out, output_crs, resolution)
        frac34 = _compute_aoi_cloud_fraction(
            dc, p["ODC_id_row34"], product, aoi_out, output_crs, resolution)
 
        # Use the worst (max) of both tiles
        fracs = [f for f in (frac33, frac34) if f is not None]
        if not fracs:
            logger.warning("  Could not compute AOI cloud for pair %s — keeping it",
                           p["SceneDate"])
            kept.append(p)
            continue
 
        worst = max(fracs)
        pct = worst * 100
 
        if worst <= threshold:
            logger.info("  PASS %s %s — AOI cloud %.1f%% (r33=%.1f%% r34=%.1f%%)",
                        p["Product"], p["SceneDate"], pct,
                        (frac33 or 0) * 100, (frac34 or 0) * 100)
            p["CloudCover_AOI_row33"] = round((frac33 or 0) * 100, 2)
            p["CloudCover_AOI_row34"] = round((frac34 or 0) * 100, 2)
            kept.append(p)
        else:
            logger.info("  REJECT %s %s — AOI cloud %.1f%% > %.1f%% threshold",
                        p["Product"], p["SceneDate"], pct,
                        cloud_cover_aoi_max)
 
    logger.info("  AOI cloud filter: %d/%d pairs kept", len(kept), len(pairs))
    return kept
 
 
def build_pairs(cands, wrs_rows):
    if not cands:
        return []
    df = pd.DataFrame(cands)
    if df.empty:
        return []
    pairs = []
    for (prod, scenedt), g in df.groupby(["Product", "SceneDT"]):
        rows_present = set(g["WRS_ROW"].tolist())
        if not wrs_rows.issubset(rows_present):
            continue
        id_row, cc_row = {}, {}
        for _, r in g.iterrows():
            rr = int(r["WRS_ROW"])
            if rr not in id_row:
                id_row[rr] = str(r["ODC_id"])
                cc_row[rr] = r["CloudCover"]
            else:
                old_cc, new_cc = cc_row[rr], r["CloudCover"]
                if old_cc is None and new_cc is not None:
                    id_row[rr], cc_row[rr] = str(r["ODC_id"]), new_cc
                elif (old_cc is not None) and (new_cc is not None) and (new_cc < old_cc):
                    id_row[rr], cc_row[rr] = str(r["ODC_id"]), new_cc
        pairs.append({
            "Product": prod, "SceneDT": scenedt,
            "SceneDate": pd.Timestamp(scenedt).date().isoformat(),
            "ODC_id_row33": id_row[33], "ODC_id_row34": id_row[34],
            "CloudCover_row33": cc_row.get(33), "CloudCover_row34": cc_row.get(34),
        })
    return pairs
 
 
def choose_best_pair(target_date, pairs, sensor_pref):
    if not pairs:
        return None
    target = _as_utc_naive(pd.Timestamp(target_date).normalize())
 
    def score(p):
        off = int((p["SceneDT"] - target).days)
        return (abs(off), _product_priority(p["Product"], sensor_pref),
                0 if off >= 0 else 1, p["Product"], p["SceneDate"],
                p["ODC_id_row33"], p["ODC_id_row34"])
 
    best = min(pairs, key=score)
    off = int((best["SceneDT"] - target).days)
    return {
        "TargetDate": pd.Timestamp(target_date).date().isoformat(),
        "Product": best["Product"], "SceneDate": best["SceneDate"],
        "OffsetDays": off, "WRS_PATH": WRS_PATH,
        "ODC_id_row33": best["ODC_id_row33"], "ODC_id_row34": best["ODC_id_row34"],
        "CloudCover_row33": best["CloudCover_row33"],
        "CloudCover_row34": best["CloudCover_row34"],
        "CloudCover_AOI_row33": best.get("CloudCover_AOI_row33"),
        "CloudCover_AOI_row34": best.get("CloudCover_AOI_row34"),
    }
 
 
def run_pair_coverage():
    dc = datacube.Datacube()
    bbox = get_bbox_wgs84()
    aoi_out = get_aoi_in_crs(OUTPUT_CRS)  # needed for AOI cloud check
    st_products = list_landsat_st_products(dc, SENSOR_PREF)
    if not st_products:
        raise RuntimeError("No Landsat ST products detected in ODC index.")
 
    logger.info("Detected Landsat ST products:")
    for p in st_products:
        logger.info("  - %s", p)
 
    sel = pd.read_csv(TARGETS_CSV, parse_dates=["Date"])
    target_dates = sorted({pd.Timestamp(d).date() for d in sel["Date"].dropna()})
    logger.info("Loaded %d target dates from %s", len(target_dates), TARGETS_CSV)
    logger.info("Searching ±%d days; Path %d, Rows %s", SEARCH_WINDOW_DAYS, WRS_PATH, sorted(WRS_ROWS))
    logger.info("Cloud filter stage 1 (full scene): %s",
                "OFF" if CLOUD_COVER_MAX is None else f"{CLOUD_COVER_MAX}%%")
    logger.info("Cloud filter stage 2 (AOI only):   %s",
                "OFF" if CLOUD_COVER_AOI_MAX is None else f"{CLOUD_COVER_AOI_MAX}%%")
 
    rows_cov, rows_eff = [], []
    for td in target_dates:
        logger.info("--- Target date: %s ---", td.isoformat())
        t0, t1 = _date_range(td, SEARCH_WINDOW_DAYS)
 
        # Stage 1: fast metadata filter
        cands = find_candidates_for_date(dc, st_products, bbox, t0, t1,
                                         WRS_PATH, WRS_ROWS, CLOUD_COVER_MAX)
        pairs = build_pairs(cands, WRS_ROWS)
        logger.info("  Candidates after stage 1: %d scenes → %d valid pairs",
                     len(cands), len(pairs))
 
        # Stage 2: precise AOI cloud filter (downloads QA_PIXEL per pair)
        pairs = filter_pairs_by_aoi_cloud(pairs, dc, aoi_out, OUTPUT_CRS,
                                           RESOLUTION, CLOUD_COVER_AOI_MAX)
 
        best = choose_best_pair(td, pairs, SENSOR_PREF)
 
        rows_cov.append({
            "target_date": td.isoformat(),
            "num_candidates_wrs_filtered": len(cands),
            "num_valid_pairs": len(pairs),
            "chosen_product":       "" if best is None else best["Product"],
            "chosen_scene_date":    "" if best is None else best["SceneDate"],
            "chosen_offset_days":   "" if best is None else best["OffsetDays"],
            "chosen_odc_id_row33":  "" if best is None else best["ODC_id_row33"],
            "chosen_odc_id_row34":  "" if best is None else best["ODC_id_row34"],
        })
        if best is not None:
            rows_eff.append(best)
 

    coverage  = pd.DataFrame(rows_cov)
    effective = pd.DataFrame(rows_eff)

    # Deduplica: si dos fechas eligieron la misma escena, queda la más cercana
    if not effective.empty:
        effective["offset_abs"] = effective["OffsetDays"].abs()
        before = len(effective)
        effective = (effective
                     .sort_values("offset_abs")
                     .drop_duplicates(subset=["SceneDate", "Product"], keep="first")
                     .drop(columns=["offset_abs"])
                     .sort_values("TargetDate")
                     .reset_index(drop=True))
        after = len(effective)
        if before != after:
            logger.info("Dedup: removed %d duplicate scene(s)", before - after)

    OUTPUTS_DIR.mkdir(exist_ok=True)

    coverage.to_csv(OUT_COVERAGE_PAIRS_CSV, index=False)
    effective.to_csv(OUT_EFFECTIVE_TILES_CSV, index=False)
 
    logger.info("Coverage report: %s", OUT_COVERAGE_PAIRS_CSV)
    logger.info("Effective tile pairs: %s (%d rows)", OUT_EFFECTIVE_TILES_CSV, len(effective))
    if effective.empty:
        logger.warning("No valid pairs found — ODC did not return BOTH rows for your window.")
 
run_pair_coverage()


2026-05-05 23:32:22  INFO      AOI bbox (EPSG:4326): {'x': (np.float64(-121.9404544080094), np.float64(-121.19670027286202)), 'y': (np.float64(37.62499087712795), np.float64(38.58916212265879))}
2026-05-05 23:32:22  INFO      Detected Landsat ST products:
2026-05-05 23:32:22  INFO        - landsat9_c2l2_st
2026-05-05 23:32:22  INFO        - landsat8_c2l2_st
2026-05-05 23:32:22  INFO        - landsat7_c2l2_st
2026-05-05 23:32:22  INFO        - landsat5_c2l2_st
2026-05-05 23:32:22  INFO      Loaded 3 target dates from inputs/target_dates.csv
2026-05-05 23:32:22  INFO      Searching ±16 days; Path 44, Rows [33, 34]
2026-05-05 23:32:22  INFO      Cloud filter stage 1 (full scene): 70%%
2026-05-05 23:32:22  INFO      Cloud filter stage 2 (AOI only):   10%%
2026-05-05 23:32:22  INFO      --- Target date: 2017-08-11 ---
2026-05-05 23:32:22  INFO        Candidates after stage 1: 8 scenes → 4 valid pairs
2026-05-05 23:32:22  INFO      Found credentials in environment variables.
2026-05-05 23:32

In [28]:
# ============================================================
#  CHUNK 7 — Scene metadata
# ============================================================

def run_scene_metadata():
    dc = datacube.Datacube()
    bbox = get_bbox_wgs84()
    EFFECTIVE_CSV = Path(CFG["outputs"]["effective_tiles_csv"])
    OUT_META_CSV  = OUTPUTS_DIR / "scene_metadata_odc_pairs.csv"

    eff = pd.read_csv(EFFECTIVE_CSV)
    if eff.empty:
        raise ValueError(f"{EFFECTIVE_CSV} is empty (run chunk 6 first).")

    required = {"TargetDate", "Product", "SceneDate", "OffsetDays", "ODC_id_row33", "ODC_id_row34"}
    missing = required - set(eff.columns)
    if missing:
        raise ValueError(f"Missing columns in {EFFECTIVE_CSV}: {missing}")

    def _safe_props(ds):
        md = getattr(ds, "metadata_doc", None) or {}
        return (md.get("properties", {}) or {}) if isinstance(md, dict) else {}

    rows = []
    for _, r in eff.iterrows():
        product    = str(r["Product"])
        scene_date = pd.Timestamp(r["SceneDate"]).normalize()
        t0, t1     = scene_date, scene_date + pd.Timedelta(days=1)
        wanted33, wanted34 = str(r["ODC_id_row33"]), str(r["ODC_id_row34"])

        dss = dc.find_datasets(product=product, time=(t0, t1), **bbox)
        if not dss:
            continue

        ds33 = ds34 = None
        for ds in dss:
            if str(ds.id) == wanted33: ds33 = ds
            elif str(ds.id) == wanted34: ds34 = ds
        if ds33 is None: ds33 = dss[0]
        if ds34 is None: ds34 = dss[-1] if len(dss) > 1 else dss[0]

        for label, ds in [("row33", ds33), ("row34", ds34)]:
            props = _safe_props(ds)
            path, row = _extract_wrs_path_row(ds)
            rows.append({
                "TargetDate": str(r["TargetDate"]),
                "Product": product,
                "SceneDate": scene_date.date().isoformat(),
                "OffsetDays": int(r["OffsetDays"]),
                "tile_label": label,
                "ODC_id": str(ds.id),
                "landsat:scene_id": props.get("landsat:scene_id"),
                "datetime": props.get("datetime"),
                #"eo:cloud_cover": props.get("eo:cloud_cover"),
                "landsat:wrs_path": path,
                "landsat:wrs_row": row,
                "CloudCover_scene": r.get(f"CloudCover_{label}"),
                "CloudCover_AOI": r.get(f"CloudCover_AOI_{label}"),
            })

    meta = pd.DataFrame(rows)
    OUTPUTS_DIR.mkdir(exist_ok=True)
    meta.to_csv(OUT_META_CSV, index=False)
    logger.info("Scene metadata (pairs): %s (%d rows)", OUT_META_CSV, len(meta))

run_scene_metadata()


2026-05-05 23:33:08  INFO      AOI bbox (EPSG:4326): {'x': (np.float64(-121.9404544080094), np.float64(-121.19670027286202)), 'y': (np.float64(37.62499087712795), np.float64(38.58916212265879))}
2026-05-05 23:33:08  INFO      Scene metadata (pairs): outputs/scene_metadata_odc_pairs.csv (6 rows)


In [29]:
# ============================================================
#  CHUNK 8A — Setup for tile loading
# ============================================================

EFFECTIVE_TILES_CSV = Path(CFG["outputs"]["effective_tiles_csv"])
OUT_DIR_TIFS  = Path(CFG["outputs"]["tifs_dir"])
OUT_ZIP       = Path(CFG["outputs"]["zip_file"])
OUT_SUMMARY   = Path(CFG["outputs"]["summary_csv"])

OUT_DIR_TIFS.mkdir(parents=True, exist_ok=True)
OUT_ZIP.parent.mkdir(parents=True, exist_ok=True)

dc = datacube.Datacube()
aoi_out = get_aoi_in_crs(OUTPUT_CRS)

eff = pd.read_csv(EFFECTIVE_TILES_CSV)
if eff.empty:
    raise ValueError(f"{EFFECTIVE_TILES_CSV} is empty (run chunk 6 first).")

required_cols = {"TargetDate","Product","SceneDate","OffsetDays","ODC_id_row33","ODC_id_row34"}
missing = required_cols - set(eff.columns)
if missing:
    raise ValueError(f"Missing columns in {EFFECTIVE_TILES_CSV}: {missing}")

logger.info("Loaded %d effective TargetDate pairs from: %s", len(eff), EFFECTIVE_TILES_CSV)
logger.info("Mask: WATER_ONLY=%s  USE_CLEAR=%s  USE_RADSAT=%s", WATER_ONLY, USE_CLEAR, USE_RADSAT_MASK)
logger.info("Output folder: %s", OUT_DIR_TIFS)


2026-05-05 23:33:08  INFO      Loaded 3 effective TargetDate pairs from: inputs/target_dates_effective_tiles.csv
2026-05-05 23:33:08  INFO      Mask: WATER_ONLY=True  USE_CLEAR=False  USE_RADSAT=True
2026-05-05 23:33:08  INFO      Output folder: outputs/mosaicos_odc_lst_delta


In [30]:
# ============================================================
#  CHUNK 8B — Tile loader helpers
# ============================================================

meas_cache = {}
 
def _get_ds_by_uuid(dc, id_str: str):
    try:
        ds_uuid = uuid.UUID(str(id_str))
    except Exception:
        raise ValueError(f"ODC_id is not a valid UUID: {id_str}")
    ds_obj = dc.index.datasets.get(ds_uuid)
    if ds_obj is None:
        raise RuntimeError(f"Could not fetch dataset from index: {id_str}")
    return ds_obj
 
def _finite_stats(da: xr.DataArray):
    v = da.values
    m = np.isfinite(v)
    n = int(m.sum())
    if n == 0:
        return 0, None, None, None
    vv = v[m]
    return n, float(vv.min()), float(vv.max()), float(vv.mean())
 
def _get_measurement_def(prod_def, band_name: str) -> dict:
    meas = prod_def.get("measurements", None)
    if meas is None:
        return {}
    if isinstance(meas, dict):
        return meas.get(band_name, {}) or {}
    if isinstance(meas, list):
        for item in meas:
            if isinstance(item, dict) and item.get("name") == band_name:
                return item
        return {}
    return {}
 
def _apply_scale_offset_to_kelvin(st_da: xr.DataArray, product: str,
                                   st_band: str, dc) -> xr.DataArray:
    st_f = st_da.astype("float32")
    scale  = st_da.attrs.get("scale_factor", None)
    offset = st_da.attrs.get("add_offset", None)
 
    prod = dc.index.products.get_by_name(product)
    mdef = {}
    if prod is not None:
        mdef = _get_measurement_def(prod.definition, st_band)
 
    if scale is None:
        scale = mdef.get("scale_factor", None)
    if offset is None:
        offset = mdef.get("add_offset", None)
 
    if (scale is not None) and (offset is not None):
        return st_f * float(scale) + float(offset)
 
    return st_f * 0.00341802 + 149.0
 
def _st_to_celsius(st_da: xr.DataArray, product: str,
                    st_band: str, dc) -> xr.DataArray:
    kelvin = _apply_scale_offset_to_kelvin(st_da, product, st_band, dc)
    return kelvin - 273.15
 
def _qa_bit_fractions(qa: xr.DataArray):
    return {
        "frac_clear":  float(_bit_is_set(qa, BIT_CLEAR).mean().values),
        "frac_water":  float(_bit_is_set(qa, BIT_WATER).mean().values),
        "frac_cloud":  float(_bit_is_set(qa, BIT_CLOUD).mean().values),
        "frac_shadow": float(_bit_is_set(qa, BIT_CLOUD_SHADOW).mean().values),
        "frac_snow":   float(_bit_is_set(qa, BIT_SNOW).mean().values),
    }
 
def _load_raw_tile(ds_obj, product: str, label: str,
                   dc, meas_cache, output_crs, resolution, use_radsat_mask):
    """Carga ST + QA crudos SIN aplicar máscara. Devuelve (st_raw, qa_raw, diag) o (None, None, diag)."""
    if product not in meas_cache:
        meas_cache[product] = resolve_measurements_from_definition(dc, product)
    st_band, qa_pixel_band, qa_radsat_band = meas_cache[product]
 
    measurements = [st_band, qa_pixel_band]
    if (qa_radsat_band is not None) and use_radsat_mask:
        measurements.append(qa_radsat_band)
 
    with Env(AWS_REQUEST_PAYER="requester", GDAL_DISABLE_READDIR_ON_OPEN="YES"):
        ds = dc.load(
            datasets=[ds_obj],
            measurements=measurements,
            output_crs=output_crs,
            resolution=resolution,
            group_by="solar_day",
            skip_broken_datasets=True,
        )
 
    if ("time" not in ds.dims) or (ds.time.size == 0):
        logger.warning("[%s] no time dimension after load", label)
        return None, None, {"label": label, "status": "no_time"}
 
    st_raw = ds[st_band].isel(time=0)
    qa_raw = ds[qa_pixel_band].isel(time=0)
 
    diag = {"label": label, "status": "ok",
            "st_band": st_band, "qa_pixel_band": qa_pixel_band,
            "qa_radsat_band": qa_radsat_band}
 
    # If radsat requested, combine it into qa info
    if use_radsat_mask and (qa_radsat_band is not None):
        radsat = ds[qa_radsat_band].isel(time=0)
        diag["_radsat"] = radsat
 
    n_total, _, _, _ = _finite_stats(st_raw)
    diag["n_total"] = n_total
    logger.info("[%s] loaded — %s pixels", label, f"{n_total:,}")
 
    return st_raw, qa_raw, diag
 



In [31]:
# ============================================================
#  CHUNK 8C + 8D — Merge RAW → mask → clip → write
#  (merge ANTES de la máscara para eliminar la franja vacía)
# ============================================================

run_rows = []
 
for _, r in eff.iterrows():
    target_date = str(r["TargetDate"])
    product     = str(r["Product"])
    scene_date  = str(r["SceneDate"])
    offset_days = int(r["OffsetDays"])
    id33        = str(r["ODC_id_row33"])
    id34        = str(r["ODC_id_row34"])
 
    logger.info("=" * 90)
    logger.info("TargetDate=%s  SceneDate=%s  Product=%s  Offset=%d",
                target_date, scene_date, product, offset_days)
 
    ds33 = _get_ds_by_uuid(dc, id33)
    ds34 = _get_ds_by_uuid(dc, id34)
 
    st_band, qa_pixel_band, qa_radsat_band = meas_cache.get(product, (None, None, None))
 
    # 1) Load raw tiles (no masking)
    st33, qa33, d33 = _load_raw_tile(ds33, product, "row33",
                                      dc, meas_cache, OUTPUT_CRS, RESOLUTION, USE_RADSAT_MASK)
    st34, qa34, d34 = _load_raw_tile(ds34, product, "row34",
                                      dc, meas_cache, OUTPUT_CRS, RESOLUTION, USE_RADSAT_MASK)
 
    # Refresh band names from cache
    st_band, qa_pixel_band, qa_radsat_band = meas_cache[product]
 
    # Check if both failed
    raw_tiles_st = [t for t in (st33, st34) if t is not None]
    raw_tiles_qa = [t for t in (qa33, qa34) if t is not None]
 
    if len(raw_tiles_st) == 0:
        logger.warning("SKIP %s: no data in BOTH tiles.", target_date)
        run_rows.append({
            "TargetDate": target_date, "SceneDate": scene_date,
            "Product": product, "OffsetDays": offset_days,
            "WRS_PATH": WRS_PATH,
            "ODC_id_row33": id33, "ODC_id_row34": id34,
            "OutTIF": "", "status": "skipped_no_data",
            "WATER_ONLY": WATER_ONLY, "USE_CLEAR": USE_CLEAR,
            "USE_RADSAT_MASK": USE_RADSAT_MASK,
            "OUTPUT_CRS": OUTPUT_CRS, "RESOLUTION": str(RESOLUTION),
        })
        continue
 
    # 2) Merge raw ST and QA across tiles FIRST
    raw_tiles_st = [t.where(np.isfinite(t.astype("float32")), np.nan).astype("float32") for t in raw_tiles_st]
    mosaic_st = merge_arrays(raw_tiles_st)
    mosaic_st = mosaic_st.rio.write_crs(OUTPUT_CRS)
 
    # For QA: use max so any "bad" bit from either tile wins
    raw_tiles_qa = [t.astype("uint16") for t in raw_tiles_qa]
    # merge_arrays takes first valid by default — for QA we want that too
    # (where one tile has data and the other doesn't, use whichever has data)
    mosaic_qa = merge_arrays(raw_tiles_qa)
    mosaic_qa = mosaic_qa.rio.write_crs(OUTPUT_CRS)
 
    # Merge radsat if used
    mosaic_radsat = None
    if USE_RADSAT_MASK:
        radsats = []
        if d33.get("_radsat") is not None:
            radsats.append(d33["_radsat"].astype("uint16"))
        if d34.get("_radsat") is not None:
            radsats.append(d34["_radsat"].astype("uint16"))
        if radsats:
            mosaic_radsat = merge_arrays(radsats)
 
    # 3) QA diagnostics on merged mosaic
    bits = _qa_bit_fractions(mosaic_qa)
    logger.info("[mosaic] QA bits: clear=%.4f water=%.4f cloud=%.4f shadow=%.4f snow=%.4f",
                bits["frac_clear"], bits["frac_water"],
                bits["frac_cloud"], bits["frac_shadow"], bits["frac_snow"])
 
    # 4) Build mask on MERGED QA
    good = build_good_mask(mosaic_qa, water_only=WATER_ONLY, use_clear=USE_CLEAR)
 
    if USE_RADSAT_MASK and (mosaic_radsat is not None):
        good = good & (mosaic_radsat == 0)
 
    # 5) Apply mask → convert to Celsius
    st_masked = mosaic_st.where(good)
    n_good, _, _, _ = _finite_stats(st_masked)
 
    st_c = _st_to_celsius(st_masked, product, st_band, dc)
    st_c = st_c.rio.write_crs(OUTPUT_CRS)
 
    # 6) Clip to AOI
    st_clip = st_c.rio.clip(aoi_out.geometry, aoi_out.crs, drop=True)
    n_clip, tmin, tmax, tmean = _finite_stats(st_clip)
 
    logger.info("[mosaic] good=%s  after_clip=%s  C(min/max/mean)=%s/%s/%s",
                f"{n_good:,}", f"{n_clip:,}", tmin, tmax, tmean)
 
    if n_clip == 0:
        logger.warning("SKIP %s: 0 valid pixels after AOI clip.", target_date)
        run_rows.append({
            "TargetDate": target_date, "SceneDate": scene_date,
            "Product": product, "OffsetDays": offset_days,
            "WRS_PATH": WRS_PATH,
            "ODC_id_row33": id33, "ODC_id_row34": id34,
            "OutTIF": "", "status": "skipped_empty_after_clip",
            "WATER_ONLY": WATER_ONLY, "USE_CLEAR": USE_CLEAR,
            "USE_RADSAT_MASK": USE_RADSAT_MASK,
            "OUTPUT_CRS": OUTPUT_CRS, "RESOLUTION": str(RESOLUTION),
        })
        continue
 
    # 7) Write
    out_tif = OUT_DIR_TIFS / f"lst_delta_{target_date}_scene_{scene_date}_{product}_P{WRS_PATH}_R33R34.tif"
 
    mosaic_out = st_clip.astype("float32")
    mosaic_out = mosaic_out.where(np.isfinite(mosaic_out), NODATA_OUT)
    mosaic_out.rio.write_nodata(NODATA_OUT, inplace=True)
    mosaic_out.rio.to_raster(out_tif, nodata=NODATA_OUT, compress="LZW")
    logger.info("Wrote: %s", out_tif)
 
    # Per-tile QA stats (from individual tiles before merge)
    bits33 = _qa_bit_fractions(qa33) if qa33 is not None else {}
    bits34 = _qa_bit_fractions(qa34) if qa34 is not None else {}
 
    prod_obj = dc.index.products.get_by_name(product)
    mdef = _get_measurement_def(prod_obj.definition, st_band) if prod_obj else {}
 
    run_rows.append({
        "TargetDate": target_date, "SceneDate": scene_date,
        "Product": product, "OffsetDays": offset_days,
        "WRS_PATH": WRS_PATH,
        "ODC_id_row33": id33, "ODC_id_row34": id34,
        "OutTIF": str(out_tif), "status": "written",
        "WATER_ONLY": WATER_ONLY, "USE_CLEAR": USE_CLEAR,
        "USE_RADSAT_MASK": USE_RADSAT_MASK,
        "OUTPUT_CRS": OUTPUT_CRS, "RESOLUTION": str(RESOLUTION),
        "mosaic_valid_px": n_clip,
        "mosaic_n_good_premask": n_good,
        "mosaic_tmin_c": tmin, "mosaic_tmax_c": tmax, "mosaic_tmean_c": tmean,
        "mosaic_frac_clear": bits.get("frac_clear"),
        "mosaic_frac_water": bits.get("frac_water"),
        "mosaic_frac_cloud": bits.get("frac_cloud"),
        "row33_frac_clear": bits33.get("frac_clear"),
        "row33_frac_water": bits33.get("frac_water"),
        "row34_frac_clear": bits34.get("frac_clear"),
        "row34_frac_water": bits34.get("frac_water"),
        "row33_n_total": d33.get("n_total"),
        "row34_n_total": d34.get("n_total"),
        "scale_def": mdef.get("scale_factor"),
        "offset_def": mdef.get("add_offset"),
        "st_band": st_band,
        "qa_pixel_band": qa_pixel_band,
        "qa_radsat_band": qa_radsat_band,
    })


2026-05-05 23:33:08  INFO      ==========================================================================================
2026-05-05 23:33:08  INFO      TargetDate=2017-08-11  SceneDate=2017-08-09  Product=landsat7_c2l2_st  Offset=-2
2026-05-05 23:33:08  INFO      Found credentials in environment variables.
2026-05-05 23:33:12  INFO      [row33] loaded — 54,206,011 pixels
2026-05-05 23:33:12  INFO      Found credentials in environment variables.
2026-05-05 23:33:14  INFO      [row34] loaded — 54,108,768 pixels
2026-05-05 23:33:15  INFO      Found credentials in environment variables.
2026-05-05 23:33:15  INFO      Found credentials in environment variables.
2026-05-05 23:33:15  INFO      Found credentials in environment variables.
2026-05-05 23:33:15  INFO      Found credentials in environment variables.
2026-05-05 23:33:15  INFO      Found credentials in environment variables.
2026-05-05 23:33:17  INFO      Found credentials in environment variables.
2026-05-05 23:33:17  INFO      Fou

In [32]:
# ============================================================
#  CHUNK 8E — Summary + ZIP
# ============================================================

summary = pd.DataFrame(run_rows)
summary.to_csv(OUT_SUMMARY, index=False)

written = summary.loc[summary["status"] == "written", "OutTIF"].tolist()

with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in written:
        if p and os.path.exists(p):
            z.write(p, arcname=os.path.basename(p))

logger.info("=" * 90)
logger.info("Run summary: %s (rows=%d, written=%d)", OUT_SUMMARY, len(summary), len(written))
logger.info("ZIP: %s", OUT_ZIP)
logger.info("Output folder: %s", OUT_DIR_TIFS)


2026-05-05 23:34:22  INFO      ==========================================================================================
2026-05-05 23:34:22  INFO      Run summary: outputs/mosaicos_odc_lst_delta/run_summary_pairs.csv (rows=3, written=3)
2026-05-05 23:34:22  INFO      ZIP: outputs/mosaicos/mosaicos_celsius_odc_delta.zip
2026-05-05 23:34:22  INFO      Output folder: outputs/mosaicos_odc_lst_delta


In [33]:
# ============================================================
#  Celda nueva — Pégala después del chunk 8E y córrela
# ============================================================

import pandas as pd
from pathlib import Path

# --- Lee los archivos que tu pipeline ya generó ---
coverage  = pd.read_csv("outputs/satellite_coverage_report_pairs.csv")
summary   = pd.read_csv("outputs/mosaicos_odc_lst_delta/run_summary_pairs.csv")

# --- Combina todo en una sola tabla ---
report = summary.copy()

# Agrega info del coverage report (candidatos y pares encontrados)
if not coverage.empty:
    cov_cols = ["target_date", "num_candidates_wrs_filtered", "num_valid_pairs"]
    cov = coverage[cov_cols].copy()
    cov = cov.rename(columns={"target_date": "TargetDate"})
    report = report.merge(cov, on="TargetDate", how="left")

# --- Calcula métricas útiles ---
report["offset_abs"] = report["OffsetDays"].abs()

# Reordena columnas para que sea legible
col_order = [
    # Identificación
    "TargetDate", "SceneDate", "OffsetDays", "offset_abs",
    "Product", "WRS_PATH",
    # Búsqueda
    "num_candidates_wrs_filtered", "num_valid_pairs",
    # Nubosidad escena completa
    "row33_frac_clear", "row33_frac_water",
    "row34_frac_clear", "row34_frac_water",
    # Píxeles
    "row33_n_after_clip", "row34_n_after_clip", "mosaic_valid_px",
    # Temperatura
    "mosaic_tmin_c", "mosaic_tmax_c", "mosaic_tmean_c",
    # Bandas y escala
    "st_band", "qa_pixel_band", "qa_radsat_band",
    "row33_scale_def", "row33_offset_def",
    "row34_scale_def", "row34_offset_def",
    # Config usada
    "WATER_ONLY", "USE_CLEAR", "USE_RADSAT_MASK",
    "OUTPUT_CRS", "RESOLUTION",
    # IDs
    "ODC_id_row33", "ODC_id_row34",
    # Resultado
    "status", "OutTIF",
]

# Solo incluye columnas que existan
col_order = [c for c in col_order if c in report.columns]
report = report[col_order]

# --- Muestra ---
print(f"Total de escenas procesadas: {len(report)}")
print(f"Escritas exitosamente: {(report['status'] == 'written').sum()}")
print(f"Saltadas: {(report['status'] != 'written').sum()}")
print()

# Tabla bonita
with pd.option_context("display.max_columns", None, "display.width", 200,
                        "display.max_colwidth", 40, "display.max_rows", None):
    display(report)

# --- Guarda como CSV ---
out_path = "outputs/full_diagnostics_report.csv"
report.to_csv(out_path, index=False)
print(f"\n✓ Guardado en: {out_path}")

Total de escenas procesadas: 3
Escritas exitosamente: 3
Saltadas: 0



,TargetDate,SceneDate,OffsetDays,offset_abs,Product,WRS_PATH,num_candidates_wrs_filtered,num_valid_pairs,row33_frac_clear,row33_frac_water,row34_frac_clear,row34_frac_water,mosaic_valid_px,mosaic_tmin_c,mosaic_tmax_c,mosaic_tmean_c,st_band,qa_pixel_band,qa_radsat_band,WATER_ONLY,USE_CLEAR,USE_RADSAT_MASK,OUTPUT_CRS,RESOLUTION,ODC_id_row33,ODC_id_row34,status,OutTIF
0,2017-08-11,2017-08-09,-2,2,landsat7_c2l2_st,44,8,4,0.535868,0.007916,0.270188,0.013308,176900,19.406830,65.991058,23.760300,lwir,qa_pixel,qa_radsat,True,False,True,EPSG:32610,"(-30, 30)",dc284f78-3788-5951-97f3-f43a3609010e,70bc246d-ac4e-5234-8dae-ff11d72e65f6,written,outputs/mosaicos_odc_lst_delta/lst_d...
1,2018-10-06,2018-10-07,1,1,landsat8_c2l2_st,44,8,3,0.698352,0.015621,0.695257,0.260603,229506,17.154388,40.164459,20.997499,lwir11,qa_pixel,qa_radsat,True,False,True,EPSG:32610,"(-30, 30)",19e83604-db94-51b3-be00-efe59cb1a605,3e22c522-1d28-5e28-8b63-e18988fdb422,written,outputs/mosaicos_odc_lst_delta/lst_d...
2,2023-09-18,2023-09-19,1,1,landsat8_c2l2_st,44,10,4,0.695109,0.014450,0.504256,0.090556,232624,19.167572,46.337433,23.483610,lwir11,qa_pixel,qa_radsat,True,False,True,EPSG:32610,"(-30, 30)",09768259-5e10-5c5e-a207-208afcbedcc5,e313d238-606d-5bb5-bede-ce2c25fef037,written,outputs/mosaicos_odc_lst_delta/lst_d...



✓ Guardado en: outputs/full_diagnostics_report.csv


In [34]:
# Diagnóstico: pick first scene
row = eff.iloc[0]
product = str(row["Product"])
ds33 = _get_ds_by_uuid(dc, str(row["ODC_id_row33"]))
ds34 = _get_ds_by_uuid(dc, str(row["ODC_id_row34"]))
st_band, qa_pixel_band, _ = resolve_measurements_from_definition(dc, product)

with Env(AWS_REQUEST_PAYER="requester", GDAL_DISABLE_READDIR_ON_OPEN="YES"):
    raw33 = dc.load(datasets=[ds33], measurements=[st_band, qa_pixel_band],
                    output_crs=OUTPUT_CRS, resolution=RESOLUTION, group_by="solar_day")
    raw34 = dc.load(datasets=[ds34], measurements=[st_band, qa_pixel_band],
                    output_crs=OUTPUT_CRS, resolution=RESOLUTION, group_by="solar_day")

print(f"Scene: {row['SceneDate']} / {product}")
print(f"Row33 y range: {float(raw33.y.min()):.1f} → {float(raw33.y.max()):.1f}")
print(f"Row34 y range: {float(raw34.y.min()):.1f} → {float(raw34.y.max()):.1f}")

gap = float(raw34.y.max()) - float(raw33.y.min())
print(f"\nOverlap/gap: {gap:.1f} m  ({'OVERLAP' if gap > 0 else 'GAP'})")

qa33 = raw33[qa_pixel_band].isel(time=0)
qa34 = raw34[qa_pixel_band].isel(time=0)
water33 = _bit_is_set(qa33, BIT_WATER)
water34 = _bit_is_set(qa34, BIT_WATER)

print(f"\nRow33 water pixels total: {int(water33.sum())}")
print(f"Row34 water pixels total: {int(water34.sum())}")
print(f"\nRow33 bottom 30 rows — water: {int(water33[-30:, :].sum())}")
print(f"Row34 top 30 rows    — water: {int(water34[:30, :].sum())}")

2026-05-05 23:34:22  INFO      Found credentials in environment variables.


Scene: 2017-08-09 / landsat7_c2l2_st
Row33 y range: 4201905.0 → 4412985.0
Row34 y range: 4043205.0 → 4253415.0

Overlap/gap: 51510.0 m  (OVERLAP)

Row33 water pixels total: 429103
Row34 water pixels total: 720065

Row33 bottom 30 rows — water: 0
Row34 top 30 rows    — water: 0
